In [16]:
import numpy as np
from tqdm import tqdm

In [17]:
TAP= 'TA'
root = '/home/young/hdd1/coco-search18/'
task_img_pair = np.load(root + f'test_{TAP}_task_img_pair.npy')
img_size = 320, 512

In [18]:
SAMPLE_PER_IMAGE =10
def get_circular_mask(center, radius, h, w):
    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((Y - center[0]) ** 2 + (X - center[1]) ** 2)
    mask = dist > radius
    return mask


def saliency2scanpath(saliency, length=6, radius=24, h=320, w=512):
    xs = [256.0]
    ys = [160.0]

    for i in range(length):
        saliency = saliency / saliency.sum()
        probs = saliency.flatten()
        pos = np.random.choice(a=np.array(range(320 * 512)), size=1, p=probs)
        pos = pos[0]
        y = pos // 512
        x = pos % 512

        mask = get_circular_mask([y, x], radius, h, w).astype(np.float32)
        saliency *= mask
        ys.append(y)
        xs.append(x)
    scanpath = {}
    scanpath['X'] = xs
    scanpath['Y'] = ys
    return scanpath


results = []
for task_img in tqdm(task_img_pair):
    task =task_img.split('_')[0]
    imgname = task_img.split('_')[1]
    uniform_map = np.ones((img_size[0], img_size[1]))
    
    for _ in range(SAMPLE_PER_IMAGE):
        scanpath = saliency2scanpath(uniform_map)
        result = {}
        result['X'] = np.array(scanpath['X'])
        result['Y'] = np.array(scanpath['Y'])
        if TAP=='TP':
            result['condition'] = 'present'
        elif TAP=='TA':
            result['condition'] = 'absent'
        result['task'] = task
        result['name'] = imgname
        result['attn_map'] = uniform_map
        results.append(result)

100%|█████████████████████████████████████████| 612/612 [06:28<00:00,  1.57it/s]


In [19]:
np.save(f'pred_{TAP}_random.npy', results)
print('file saved to disk')

file saved to disk
